In [1]:
import neuron
import time
from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
sound_file = 'Alarm03.wav'

In [2]:
# h.load_file("cAD_ltb.hoc")

In [3]:
# morphology_dir = "morphologies"
# morphology_name = "TC_jy160728_A_idA.asc"
# TCcAD = h.cAD_ltb(morphology_dir, morphology_name)

In [4]:
# IN_morphology_name = "IN_jy190506_A_idC.asc"
# IN = h.cAD_ltb(morphology_dir, IN_morphology_name)

In [5]:
# h.topology()

In [6]:
# import neurom
# import neurom.viewer
# TCcAD_morph_path = 'C:/Users/yijan/OneDrive/Desktop/Interneuron_model/raw_model/morphologies/TC_jy160728_A_idA.asc'
# IN_morph_path = 'C:/Users/yijan/OneDrive/Desktop/Interneuron_model/raw_model/morphologies/IN_jy190506_A_idC.asc'
# morph_TCcAD = neurom.load_neuron(TCcAD_morph_path)
# morph_IN = neurom.load_neuron(IN_morph_path)

In [7]:
# '''
# %matplotlib notebook
# neurom.viewer.draw(morph_TCcAD, mode = '3d')
# neurom.viewer.draw(morph_IN, mode = '3d')
# '''

# %matplotlib inline
# neurom.viewer.draw(morph_TCcAD, mode = 'dendrogram')
# neurom.viewer.draw(morph_IN, mode = 'dendrogram')

In [2]:
class Cell:
    def __init__(self, gid, x, y, z, theta):
        self._gid = gid
        self._setup_cell()
        self.x = self.y = self.z = 0                     
        h.define_shape()
        self._rotate_z(theta)                                   
        self._set_position(x, y, z)  
        
        # record spikes, soma/prox-dend from all cells
        self._spike_detector = h.NetCon(self.model.soma[0](0.5)._ref_v, None, sec=self.model.soma[0])
        self.spike_times = h.Vector()
        self._spike_detector.record(self.spike_times)
        self.soma_v = h.Vector().record(self.model.soma[0](0.5)._ref_v)
        self.dend_prox_v = h.Vector().record(self.model.dend[0](0.1)._ref_v)
    
        
    def __repr__(self):
        return '{}[{}]'.format(self.name, self._gid)
    
    def _set_position(self, x, y, z):
        for sec in self.model.all:
            for i in range(sec.n3d()):
                sec.pt3dchange(i,
                               x - self.x + sec.x3d(i),
                               y - self.y + sec.y3d(i),
                               z - self.z + sec.z3d(i),
                              sec.diam3d(i))
        self.x, self.y, self.z = x, y, z
    def _rotate_z(self, theta):
        """Rotate the cell about the Z axis."""
        for sec in self.model.all:
            for i in range(sec.n3d()):
                x = sec.x3d(i)
                y = sec.y3d(i)
                c = h.cos(theta)
                s = h.sin(theta)
                xprime = x * c - y * s
                yprime = x * s + y * c
                sec.pt3dchange(i, xprime, yprime, sec.z3d(i), sec.diam3d(i))
    
class IN(Cell):
    name = 'Interneuron'
    def _setup_cell(self):
        h.load_file("cAD_ltb.hoc")
        morphology_dir = "morphologies"
        morphology_name = "IN_jy190506_A_idC.asc"
        self.model = h.cAD_ltb(morphology_dir, morphology_name)
        
        # contains all spike detectors
        self.ncs = []

        # recording of release events in distal dendrites
        self._release_detector = h.NetCon(self.model.dend[7](0.9)._ref_v, None, sec=self.model.dend[7])
        self.release_times = h.Vector()
        self._release_detector.record(self.release_times)
        # contains all release detectors
        self.ncs2 = []
        self.dend_dist_v = h.Vector().record(self.model.dend[7](0.9)._ref_v)
        self.dend_dist2_v = h.Vector().record(self.model.dend[17](0.9)._ref_v)
        
        # add synapses received from brainstem Br
        self.syn_Brprox = h.Exp2Syn(self.model.dend[0](0.1))
        self.syn_Brprox.tau1 = 1.6
        self.syn_Brprox.tau2 = 3.6
        self.syn_Brprox.e = 10
        
        self.syn_Brdist = h.Exp2Syn(self.model.dend[7](0.9))
        self.syn_Brdist.tau1 = 0.3
        self.syn_Brdist.tau2 = 2
        self.syn_Brprox.e = 10
        
        # add current clamp to be able to inject current from all my cells
        self.stim = h.IClamp(self.model.soma[0](0.5))
        self.stim.dur = 50
        self.stim.delay = 0
        self.stim.amp = 0.005
        self.stim_current = h.Vector()
        self.stim_current.record(self.stim._ref_i)

class TC(Cell):
    name = 'TC'
    def _setup_cell(self):
        h.load_file("cAD_ltb.hoc")
        morphology_dir = "morphologies"
        morphology_name = "TC_jy160728_A_idA.asc"
        self.model = h.cAD_ltb(morphology_dir, morphology_name)
        
        # add synapses received from brainstem Br
        self.syn_Br = h.Exp2Syn(self.model.dend[0](0.1))
        self.syn_Br.tau1 = 0.2
        self.syn_Br.tau2 = 1.2
        self.syn_Br.e = 1
        
        # add synapses received from IN
        self.syn_INaxonal = h.Exp2Syn(self.model.dend[0](0.1))
        self.syn_INaxonal.tau1 = 0.7
        self.syn_INaxonal.tau2 = 4.2
        self.syn_INaxonal.e = -80
        
        self.syn_INdendritic = h.Exp2Syn(self.model.dend[0](0.1))
        self.syn_INdendritic.tau1 = 0.7
        self.syn_INdendritic.tau2 = 4.2
        self.syn_INdendritic.e = -80
        
        # add current clamp to be able to inject current from all my cells
        self.stim = h.IClamp(self.model.soma[0](0.5))
        self.stim.dur = 50
        self.stim.delay = 0
        self.stim.amp = 0.225
        self.stim_current = h.Vector()
        self.stim_current.record(self.stim._ref_i)


In [3]:
class Ring:
    """
    A network of TC and IN cells where IN makes an
    axonal and dendritic synapse onto all other TCs and all cells 
    receive Br input
    
    """
    def __init__(self, N=5, r=100, PSim = None):
        """
        :param N: Number of cells. 5 by default
        :param r: radius of the network. 100um by default

        """
        if PSim is not None:
            self.PSim = PSim
        else:
            self.PSim = [350]
        
        # Add simple Br spikes 
        self._netstim = h.NetStim()
        # record simple Br spikes
        self._spike_detector = h.NetCon(self._netstim, None)
        self.spike_times = h.Vector()
        self._spike_detector.record(self.spike_times)
        # Br simple input parameters
        self._netstim.number = 10
        self._netstim.start = 1
        self._netstim.interval = 3
        self.ncBrs = []
        
        # Add Br spikes of nonstationary poisson process
        self._vecstim = h.VecStim()
        self.train_vec = h.Vector(PSim)
        self._vecstim.play(self.train_vec)
        self.ncBrsPsim = []
        
        # create TCs and IN
        self._create_cells(N, r)
        
        # create spike detectors from IN and Br to looped TC targets
        self._add_IN_syn()
        self._add_Br_syn()
        self._add_Br_Psim_syn()
        
        # create spike detectors from Br for IN targets (prox and dist denrites)
        self.ncBr2 = h.NetCon(self._netstim, self.IN_1.syn_Brprox)
        self.ncBr2.weight[0] =0.0008
        self.ncBr2.delay = 1

        self.ncBr3 = h.NetCon(self._netstim, self.IN_1.syn_Brdist)
        self.ncBr3.weight[0] =0.004
        self.ncBr3.delay = 1
        
        self.ncBrPsim2 = h.NetCon(self._vecstim, self.IN_1.syn_Brprox)
        self.ncBrPsim2.weight[0] =0.0008
        self.ncBrPsim2.delay = 1

        self.ncBrPsim3 = h.NetCon(self._vecstim, self.IN_1.syn_Brdist)
        self.ncBrPsim3.weight[0] =0.004
        self.ncBrPsim3.delay = 1
    
        
    def _create_cells(self, N, r):
        self.TCcells = [] 
        self.IN_1 = IN(0,0,0,0,0)    
        for i in range(N):
            theta = i * 2 * h.PI / N
            self.TCcells.append(TC(i, h.cos(theta) * r, h.sin(theta) * r, 0, theta))
        
    def _add_IN_syn(self):
        for target in self.TCcells:
            self.nc = h.NetCon(self.IN_1.model.soma[0](0.5)._ref_v, target.syn_INaxonal, sec=self.IN_1.model.soma[0])
            self.nc.weight[0] = 0.008
            self.nc.delay = 1
            self.IN_1.ncs.append(self.nc)
    
        for target in self.TCcells:
            self.nc2 = h.NetCon(self.IN_1.model.dend[7](0.9)._ref_v, target.syn_INdendritic, sec=self.IN_1.model.dend[7])
            self.nc2.weight[0] = 0.008
            self.nc2.delay = 1
            self.nc2.threshold = -15 # threshold needs to be lower for distal dendritic release
            self.IN_1.ncs2.append(self.nc2)
            
    def _add_Br_syn(self):
        for target in self.TCcells:
            self.ncBr = h.NetCon(self._netstim, target.syn_Br)
            self.ncBr.weight[0] = 0.04
            self.ncBr.delay = 1
            self.ncBrs.append(self.ncBr)
            
    def _add_Br_Psim_syn(self):
        for target in self.TCcells:
            self.ncBrPsim = h.NetCon(self._vecstim, target.syn_Br)
            self.ncBrPsim.weight[0] = 0.04
            self.ncBrPsim.delay = 1
            self.ncBrsPsim.append(self.ncBrPsim)

In [4]:
def createRate(d, t):
    import math
    rate = []
    for t in range(0,t):
        w = 0.85
        #lbkg(1-w) = 36.8
        lbkg = 36.8/0.15
        #lspot(1-w) = 56.5
        lspot = 56.5/0.15
        a1 = 0.62
        a2 = 1.26
        tau1 = 10
        tau2 = 22
        alpha = 12
        beta = 11.26
        ts = 0
        
        # Gprime = lbkg(1-w)+(lspot-lbkg)(1-math.exp(-d**2/(4*a1**2))-w(1-math.exp(-d**2/(4*a2**2))))
        Gprime = 36.8 + (lspot-lbkg)*(1-math.exp(-d**2/(4*a1**2))-w*(1-math.exp(-d**2/(4*a2**2))))

        if Gprime<0:
            thet = 0
        elif Gprime>0:
            thet = 1

        G = Gprime*thet

        if (t-ts)<=0:
            thet2 = 0
        elif (t-ts)>0:
            thet2 = 1

        F = thet2*alpha*(1-math.exp(-(t-ts)/tau1))-beta*(1-math.exp(-(t-ts)/tau2))
        R = G*F
        rate.append(R)
    rate_array = np.asarray(rate)
    return rate_array
    

In [5]:
def createPoissonInputs(rate_array, t):
    import NeuroTools
    import math
    from NeuroTools import stgen
    # NOTE: HAD TO UPDATE STGEN.PY FILE BECAUSE LATEST VERSION OF NEUROTOOLS IS NOT COMPATIBLE WITH LATEST VERISON OF PYTHON/NUMPY
    # specifically, I had to change xrange() to range() and add () to print. Also update relative import paths to the signals folder
    # I also had to change remove all float numbers to integers and add .astype(int) to a few lines for numpy compatibility 
    st_gen = stgen.StGen()
    intervals=np.arange(0,t)
    t_stop = t
    PSim = st_gen.inh_poisson_generator(rate_array,intervals, t_stop, array = True) # important to have rate in Hz and all other times in ms
    return PSim

In [6]:
def create2Rings(N, r, ringPSim, ring2PSim, INarrangement):
    
    ring = Ring(N,r,ringPSim)
    ring2 = Ring(N,r,ring2PSim)
    
    # connect two rings by adding IN-IN axons and dendrites
    syn_IN1axon_IN2 = h.Exp2Syn(ring2.IN_1.model.dend[0](0.1))
    syn_IN1axon_IN2.tau1 = 0.7
    syn_IN1axon_IN2.tau2 = 4.2
    syn_IN1axon_IN2.e = -80
    nc_IN1axon_IN2 = h.NetCon(ring.IN_1.model.soma[0](0.5)._ref_v, syn_IN1axon_IN2, sec=ring.IN_1.model.soma[0])
    nc_IN1axon_IN2.weight[0] = 0.006
    nc_IN1axon_IN2.delay = 1

    syn_IN2axon_IN1 = h.Exp2Syn(ring.IN_1.model.dend[0](0.1))
    syn_IN2axon_IN1.tau1 = 0.7
    syn_IN2axon_IN1.tau2 = 4.2
    syn_IN2axon_IN1.e = -80
    nc_IN2axon_IN1 = h.NetCon(ring2.IN_1.model.soma[0](0.5)._ref_v, syn_IN2axon_IN1, sec=ring2.IN_1.model.soma[0])
    nc_IN2axon_IN1.weight[0] = 0.006
    nc_IN2axon_IN1.delay = 1

    syn_IN1dend_IN2 = h.Exp2Syn(ring2.IN_1.model.dend[0](0.1))
    syn_IN1dend_IN2.tau1 = 0.7
    syn_IN1dend_IN2.tau2 = 4.2
    syn_IN1dend_IN2.e = -80
    nc_IN1dend_IN2 = h.NetCon(ring.IN_1.model.dend[17](0.9)._ref_v, syn_IN1dend_IN2, sec=ring.IN_1.model.dend[17])
    nc_IN1dend_IN2.weight[0] = 0.006
    nc_IN1dend_IN2.delay = 1
    nc_IN1dend_IN2.threshold = -15

    syn_IN2dend_IN1 = h.Exp2Syn(ring.IN_1.model.dend[0](0.1))
    syn_IN2dend_IN1.tau1 = 0.7
    syn_IN2dend_IN1.tau2 = 4.2
    syn_IN2dend_IN1.e = -80
    nc_IN2dend_IN1 = h.NetCon(ring2.IN_1.model.dend[17](0.9)._ref_v, syn_IN2dend_IN1, sec=ring2.IN_1.model.dend[17])
    nc_IN2dend_IN1.weight[0] = 0.006
    nc_IN2dend_IN1.delay = 1
    nc_IN2dend_IN1.threshold = -15  
    
    if INarrangement == 'diff':
        # add Br inputs to other INs
        synBr1toIN2 = h.Exp2Syn(ring2.IN_1.model.dend[0](0.1))
        synBr1toIN2.tau1 = 1.6 * ms
        synBr1toIN2.tau2 = 3.6 * ms
        synBr1toIN2.e = 10
        nc_Br1toIN2 = h.NetCon(ring._vecstim, synBr1toIN2)
        nc_Br1toIN2.delay = 1 * ms
        nc_Br1toIN2.weight[0] = 0.02

        synBr1toIN1 = h.Exp2Syn(ring.IN_1.model.dend[17](0.9))
        synBr1toIN1.tau1 = 0.3 * ms
        synBr1toIN1.tau2 = 2 * ms
        synBr1toIN1.e = 10
        nc_Br1toIN1 = h.NetCon(ring._vecstim, synBr1toIN1)
        nc_Br1toIN1.delay = 1 * ms
        nc_Br1toIN1.weight[0] = 0.02

        synBr2toIN1 = h.Exp2Syn(ring.IN_1.model.dend[0](0.1))
        synBr2toIN1.tau1 = 1.6 * ms
        synBr2toIN1.tau2 = 3.6 * ms
        synBr2toIN1.e = 10
        nc_Br2toIN1 = h.NetCon(ring2._vecstim, synBr2toIN1)
        nc_Br2toIN1.delay = 1 * ms
        nc_Br2toIN1.weight[0] = 0.02

        synBr2toIN2 = h.Exp2Syn(ring2.IN_1.model.dend[17](0.9))
        synBr2toIN2.tau1 = 0.3 * ms
        synBr2toIN2.tau2 = 2 * ms
        synBr2toIN2.e = 10
        nc_Br2toIN2 = h.NetCon(ring2._vecstim, synBr2toIN2)
        nc_Br2toIN2.delay = 1 * ms
        nc_Br2toIN2.weight[0] = 0.02
    elif INarrangement == 'same':
        # add Br inputs to own IN
        synBr1toIN1 = h.Exp2Syn(ring.IN_1.model.dend[0](0.1))
        synBr1toIN1.tau1 = 1.6 * ms
        synBr1toIN1.tau2 = 3.6 * ms
        synBr1toIN1.e = 10 
        nc_Br1toIN1 = h.NetCon(ring._vecstim, synBr1toIN1)
        nc_Br1toIN1.delay = 1 * ms
        nc_Br1toIN1.weight[0] = 0.02

        synBr1toIN2 = h.Exp2Syn(ring2.IN_1.model.dend[17](0.9))
        synBr1toIN2.tau1 = 0.3 * ms
        synBr1toIN2.tau2 = 2 * ms
        synBr1toIN2.e = 10
        nc_Br1toIN2 = h.NetCon(ring._vecstim, synBr1toIN2)
        nc_Br1toIN2.delay = 1 * ms
        nc_Br1toIN2.weight[0] = 0.02

        synBr2toIN2 = h.Exp2Syn(ring2.IN_1.model.dend[0](0.1))
        synBr2toIN2.tau1 = 1.6 * ms
        synBr2toIN2.tau2 = 3.6 * ms
        synBr2toIN2.e = 10
        nc_Br2toIN2 = h.NetCon(ring2._vecstim, synBr2toIN2)
        nc_Br2toIN2.delay = 1 * ms
        nc_Br2toIN2.weight[0] = 0.02

        synBr2toIN1 = h.Exp2Syn(ring.IN_1.model.dend[17](0.9))
        synBr2toIN1.tau1 = 0.3 * ms
        synBr2toIN1.tau2 = 2 * ms
        synBr2toIN1.e = 10
        nc_Br2toIN1 = h.NetCon(ring2._vecstim, synBr2toIN1)
        nc_Br2toIN1.delay = 1 * ms
        nc_Br2toIN1.weight[0] = 0.02    
    else:
        raise ValueError("INarrangment must be 'diff' or 'same'")
    
    return ring, ring2
    

In [7]:
# plt.figure()
# plt.vlines(ring.IN_1.release_times,4,5)
# plt.vlines(ring.IN_1.spike_times,3,4)
# plt.vlines(ring2.IN_1.release_times,6,7)
# plt.vlines(ring2.IN_1.spike_times,5,6)
# plt.vlines(ring.TCcells[0].spike_times,7,8)
# # plt.vlines(ring.spike_times,1,2)
# # plt.vlines(ring2.spike_times,2,3)
# plt.show()



In [7]:
ring1_PSimarray = []
ring2_PSimarray = []
for d in np.arange(0,1,0.5):
    # first make ring 1 spot inputs. 100ms delay, first 500ms is full field and then next 500ms is the spot of diameter d
    t = 20
    rate_array = createRate(d, t)
    PSim = createPoissonInputs(rate_array, t)
    PSim_delay = PSim + 30

    t = 20
    d = 0 # full field
    rate_array = createRate(d,t)
    PSim = createPoissonInputs(rate_array, t)
    fullfield_delay = PSim + 10

    ring1_PSim = np.hstack((fullfield_delay, PSim_delay))

    # make ring2 full field. 400ms
    t = 40
    d = 0
    rate_array = createRate(d, t)
    PSim = createPoissonInputs(rate_array, t)
    ring2_PSim = PSim + 10
    
    ring1_PSimarray.append(ring1_PSim)
    ring2_PSimarray.append(ring2_PSim)
    

In [8]:
ring1_input = np.array(ring1_PSimarray)
ring2_input = np.array(ring2_PSimarray)
print(ring1_input.shape)
print(ring2_input.shape)
print(ring1_input)
print(ring2_input)

np.save('ring1.npy', ring1_input, allow_pickle = True)
np.save('ring2.npy', ring2_input, allow_pickle = True)


# # Write the array to disk
# with open("ring1_spike_times.txt", 'w') as outfile:
#     for data_slice in ring1_input:
#         np.savetxt(outfile, data_slice, fmt='%-12.8f', newline=" ")
#         outfile.write('# New diameter\n')    
        
# # Write the array to disk
# with open("ring2_spike_times.txt", 'w') as outfile:
#     for data_slice in ring2_input:
#         np.savetxt(outfile, data_slice, fmt='%-12.8f', newline=" ")
#         outfile.write('# New diameter\n')      


(2,)
(2,)
[array([15.04597795, 42.39039662, 46.76200812])
 array([20.31503443, 24.96475888, 28.4135429 , 37.77441159, 42.20675968,
       43.02121552, 44.87554291, 45.40727319, 45.75214549, 48.15908273,
       48.34590783])]
[array([16.80728001, 42.21344717])
 array([12.09683959, 16.24346863, 21.47286535, 24.67934402, 25.50423756,
       30.84586636, 32.2868238 ])]
[array([15.04597795, 42.39039662, 46.76200812])
 array([20.31503443, 24.96475888, 28.4135429 , 37.77441159, 42.20675968,
       43.02121552, 44.87554291, 45.40727319, 45.75214549, 48.15908273,
       48.34590783])]
[array([16.80728001, 42.21344717])
 array([12.09683959, 16.24346863, 21.47286535, 24.67934402, 25.50423756,
       30.84586636, 32.2868238 ])]


In [10]:
new_ring1 = np.load('ring1.npy', allow_pickle = True)
new_ring2 = np.load('ring2.npy', allow_pickle = True)
print(new_ring1.shape)
print(new_ring2.shape)
print(new_ring1)
print(new_ring2)

(2,)
(2,)
[array([15.04597795, 42.39039662, 46.76200812])
 array([20.31503443, 24.96475888, 28.4135429 , 37.77441159, 42.20675968,
       43.02121552, 44.87554291, 45.40727319, 45.75214549, 48.15908273,
       48.34590783])]
[array([16.80728001, 42.21344717])
 array([12.09683959, 16.24346863, 21.47286535, 24.67934402, 25.50423756,
       30.84586636, 32.2868238 ])]


In [9]:
len(new_ring1)

2

In [15]:
# shape_window = h.PlotShape(True)
# shape_window.show(0)

1.0

In [10]:
# testing why the very first sim run of diffIN looks weird
diffIN = []
for x in range(0,len(new_ring1)):
    INarrangement = 'diff'
    ring, ring2 = create2Rings(1, 100, new_ring1[x,], new_ring2[x,], INarrangement)
    t = h.Vector().record(h._ref_t)
    ring._netstim.number = 0
    ring2._netstim.number = 0
    h.finitialize(-65 * mV)
    h.celsius = 34
    h.continuerun(50 * ms)
    diffvertstack = []
    diffvertstack = np.vstack((
    ring.IN_1.soma_v,
    ring.IN_1.dend_prox_v,
    ring.IN_1.dend_dist_v,
    ring.IN_1.dend_dist2_v,
    ring.TCcells[0].soma_v,
    #     ring.TCcells[1].soma_v,
    #     ring.TCcells[2].soma_v,
    #     ring.TCcells[3].soma_v,
    #     ring.TCcells[4].soma_v,
    ring2.IN_1.soma_v,
    ring2.IN_1.dend_prox_v,
    ring2.IN_1.dend_dist_v,
    ring2.IN_1.dend_dist2_v,
    ring2.TCcells[0].soma_v))
    #     ring2.TCcells[1].soma_v,
    #     ring2.TCcells[2].soma_v,
    #     ring2.TCcells[3].soma_v,
    #     ring2.TCcells[4].soma_v))
    diffIN.append(diffvertstack)

diffIN = np.array(diffIN)
print(diffIN.shape)
# Write the array to disk
with open('diffIN.txt', 'w') as outfile:
    outfile.write('# Array shape: {0}\n'.format(diffIN.shape))
    for data_slice in diffIN:
        np.savetxt(outfile, data_slice, fmt='%-12.8f')
        outfile.write('# New diameter\n')      

Audio(sound_file, autoplay=True)

(2, 10, 2001)


In [9]:
sameIN = []
for y in range(0,len(new_ring1)):
    INarrangement = 'same'
    ring, ring2 = create2Rings(1, 100, new_ring1[y,], new_ring2[y,], INarrangement)
    t = h.Vector().record(h._ref_t)
    ring._netstim.number = 0
    ring2._netstim.number = 0
    h.finitialize(-65 * mV)
    h.celsius = 34
    h.continuerun(50 * ms)
    samevertstack = []
    samevertstack = np.vstack((
    ring.IN_1.soma_v,
    ring.IN_1.dend_prox_v,
    ring.IN_1.dend_dist_v,
    ring.IN_1.dend_dist2_v,
    ring.TCcells[0].soma_v,
#     ring.TCcells[1].soma_v,
#     ring.TCcells[2].soma_v,
#     ring.TCcells[3].soma_v,
#     ring.TCcells[4].soma_v,
    ring2.IN_1.soma_v,
    ring2.IN_1.dend_prox_v,
    ring2.IN_1.dend_dist_v,
    ring2.IN_1.dend_dist2_v,
    ring2.TCcells[0].soma_v))
#     ring2.TCcells[1].soma_v,
#     ring2.TCcells[2].soma_v,
#     ring2.TCcells[3].soma_v,
#     ring2.TCcells[4].soma_v))
    sameIN.append(samevertstack)  

sameIN = np.array(sameIN)
print(sameIN.shape)
# Write the array to disk
with open('sameIN.txt', 'w') as outfile:
    outfile.write('# Array shape: {0}\n'.format(sameIN.shape))
    for data_slice in sameIN:
        np.savetxt(outfile, data_slice, fmt='%-12.8f')
        outfile.write('# New diameter\n')
        

Audio(sound_file, autoplay=True)

(2, 10, 2001)
